# Actividad 2.3 — Validación estructural y semántica de datos

## Objetivo
En este notebook trabajaremos la tercera etapa del pipeline de datos: la validación estructural y semántica.

A diferencia de las etapas anteriores, aquí no volveremos a realizar la ingesta ni la transformación, ya que ese proceso ya fue ejecutado previamente y el dataset resultante se encuentra en la carpeta `data/processed/`.

## Flujo del pipeline trabajado
1. Ingesta de datos
2. Limpieza y transformación
3. Validación estructural y semántica ← etapa actual

## Meta de esta actividad
- Leer el dataset procesado
- Aplicar validaciones estructurales
- Aplicar validaciones semánticas
- Separar registros válidos e inválidos
- Registrar observaciones en logs
- Exportar resultados para la siguiente etapa del pipeline

## Contexto de trabajo

En este proyecto ya existe una etapa previa de limpieza y transformación. Por lo tanto, este notebook asume que el dataset de entrada ya fue generado y almacenado en la carpeta:

`../data/processed/`

A partir de ese archivo, se verificará si los datos cumplen con reglas técnicas y lógicas antes de ser cargados posteriormente a una base de datos.

## Carga del dataset procesado

En esta etapa leeremos el archivo generado en la fase anterior del pipeline.

Si tu archivo tiene otro nombre, debes ajustar la ruta en la siguiente celda.

In [1]:
import pandas as pd
import os
import logging
import numpy as np

ruta_entrada = "../data/processed/fraude_clean.csv"

df = pd.read_csv(ruta_entrada)
df


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,trans_year,edad
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,...,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0,2020,52
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,...,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0,2020,30
2,2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,...,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0,2020,50
3,3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,...,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061,0,2020,33
4,4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,...,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734,0,2020,65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
555714,555714,2020-12-31 23:59:07,30560609640617,fraud_Reilly and Sons,health_fitness,43.77,Michael,Olson,M,558 Michael Estates,...,519,Town planner,1966-02-13,9b1f753c79894c9f4b71f04581835ada,1388534347,39.946837,-91.333331,0,2020,54
555715,555715,2020-12-31 23:59:09,3556613125071656,fraud_Hoppe-Parisian,kids_pets,111.84,Jose,Vasquez,M,572 Davis Mountains,...,28739,Futures trader,1999-12-27,2090647dac2c89a1d86c514c427f5b91,1388534349,29.661049,-96.186633,0,2020,21
555716,555716,2020-12-31 23:59:15,6011724471098086,fraud_Rau-Robel,kids_pets,86.88,Ann,Lawson,F,144 Evans Islands Apt. 683,...,3684,Musician,1981-11-29,6c5b7c8add471975aa0fec023b2e8408,1388534355,46.658340,-119.715054,0,2020,39
555717,555717,2020-12-31 23:59:24,4079773899158,fraud_Breitenberg LLC,travel,7.99,Eric,Preston,M,7020 Doyle Stream Apt. 951,...,129,Cartographer,1965-12-15,14392d723bb7737606b2700ac791b7aa,1388534364,44.470525,-117.080888,0,2020,55


## Exploración inicial

Antes de validar, revisaremos:
- estructura general del dataset
- tipos de datos
- posibles nulos remanentes

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 555719 entries, 0 to 555718
Data columns (total 25 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Unnamed: 0             555719 non-null  int64  
 1   trans_date_trans_time  555719 non-null  str    
 2   cc_num                 555719 non-null  int64  
 3   merchant               555719 non-null  str    
 4   category               555719 non-null  str    
 5   amt                    555719 non-null  float64
 6   first                  555719 non-null  str    
 7   last                   555719 non-null  str    
 8   gender                 555719 non-null  str    
 9   street                 555719 non-null  str    
 10  city                   555719 non-null  str    
 11  state                  555719 non-null  str    
 12  zip                    555719 non-null  int64  
 13  lat                    555719 non-null  float64
 14  long                   555719 non-null  float64

In [3]:
df.isnull().sum()

Unnamed: 0               0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
trans_year               0
edad                     0
dtype: int64

## Configuración del log

Crearemos un archivo de log para dejar trazabilidad del proceso de validación.

In [4]:
os.makedirs("../logs", exist_ok=True)

logging.basicConfig(
    filename="../logs/validation.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Inicio del proceso de validación del dataset procesado")
print("Logger configurado correctamente")

Logger configurado correctamente


## Reglas de validación - Detección de Fraude

Aplicaremos dos tipos de validación específicas para detección de fraude:

### Validaciones estructurales
1. `trans_date_trans_time`: Fecha de transacción válida
2. `dob`: Fecha de nacimiento válida
3. `amt`: Monto debe ser numérico
4. `gender`: Género debe ser M o F
5. `is_fraud`: Debe ser 0 o 1

### Validaciones semánticas
1. El monto `amt` debe ser mayor a 0 (transacción válida)
2. La edad calculada desde `dob` debe estar entre 18 y 120 años
3. `lat` (latitud titular): Debe estar entre -90 y 90
4. `long` (longitud titular): Debe estar entre -180 y 180

## Aplicación de validaciones

Generaremos columnas booleanas para identificar si cada registro cumple o no con cada regla.

### Nota técnica: ¿Por qué validación vectorizada?

En lugar de usar funciones con `.apply()` (que procesa fila por fila), usamos **operaciones vectorizadas** de pandas/numpy que procesan la columna completa a nivel de C. Esto es **10-50x más rápido** con datasets grandes de fraude (cientos de miles de transacciones), mejorando significativamente el rendimiento sin perder claridad.

In [5]:
df_validacion = df.copy()

# Validaciones estructurales (VECTORIZADAS - MÁS RÁPIDO)
df_validacion["val_trans_date"] = pd.to_datetime(df_validacion["trans_date_trans_time"], errors='coerce').notna()
df_validacion["val_dob"] = pd.to_datetime(df_validacion["dob"], errors='coerce').notna()
df_validacion["val_amt"] = pd.to_numeric(df_validacion["amt"], errors='coerce').notna()
df_validacion["val_gender"] = df_validacion["gender"].str.upper().isin({"M", "F"})
df_validacion["val_is_fraud"] = df_validacion["is_fraud"].astype(str).isin({"0", "1"})

# Validaciones semánticas (VECTORIZADAS)
df_validacion["val_amt_semantica"] = pd.to_numeric(df_validacion["amt"], errors='coerce') > 0

# Edad vectorizada
dob = pd.to_datetime(df_validacion["dob"], errors='coerce')
trans_date = pd.to_datetime(df_validacion["trans_date_trans_time"], errors='coerce')
edad = (trans_date.dt.year - dob.dt.year) - ((trans_date.dt.month < dob.dt.month) | ((trans_date.dt.month == dob.dt.month) & (trans_date.dt.day < dob.dt.day)))
df_validacion["val_edad"] = (edad >= 18) & (edad <= 120)

# Rangos coordenadas (VECTORIZADOS)
df_validacion["val_lat_rango"] = (pd.to_numeric(df_validacion["lat"], errors='coerce').between(-90, 90))
df_validacion["val_long_rango"] = (pd.to_numeric(df_validacion["long"], errors='coerce').between(-180, 180))

df_validacion.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,edad,val_trans_date,val_dob,val_amt,val_gender,val_is_fraud,val_amt_semantica,val_edad,val_lat_rango,val_long_rango
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,...,52,True,True,True,True,True,True,True,True,True
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,...,30,True,True,True,True,True,True,True,True,True
2,2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,...,50,True,True,True,True,True,True,True,True,True
3,3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,...,33,True,True,True,True,True,True,True,True,True
4,4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,...,65,True,True,True,True,True,True,True,True,True


## Consolidación de la validación

Un registro será considerado válido solo si cumple todas las reglas de fraude definidas.

In [ ]:
columnas_validacion = [
    "val_trans_date",
    "val_dob",
    "val_amt",
    "val_gender",
    "val_is_fraud",
    "val_amt_semantica",
    "val_edad",
    "val_lat_rango",
    "val_long_rango"
]

# Un registro es válido si pasa TODAS las validaciones
df_validacion["registro_valido"] = df_validacion[columnas_validacion].all(axis=1)

# Mostrar resumen de validación
df_validacion[["registro_valido"] + columnas_validacion].head()

,registro_valido,val_trans_date,val_dob,val_amt,val_gender,val_is_fraud,val_amt_semantica,val_edad,val_lat_rango,val_long_rango
0,True,True,True,True,True,True,True,True,True,True
1,True,True,True,True,True,True,True,True,True,True
2,True,True,True,True,True,True,True,True,True,True
3,True,True,True,True,True,True,True,True,True,True
4,True,True,True,True,True,True,True,True,True,True


## Generación de observaciones

Para cada registro inválido, documentamos qué validaciones fallaron.

In [9]:
def obtener_observaciones(row):
    errores = []

    # Validaciones estructurales
    if not row["val_trans_date"]:
        errores.append("Fecha transacción inválida")
    if not row["val_dob"]:
        errores.append("Fecha nacimiento inválida")
    if not row["val_amt"]:
        errores.append("Monto no numérico")
    if not row["val_gender"]:
        errores.append("Género no es M o F")
    if not row["val_is_fraud"]:
        errores.append("is_fraud no es 0 o 1")

    # Validaciones semánticas
    if row["val_amt"] and not row["val_amt_semantica"]:
        errores.append("Monto <= 0")
    if not row["val_edad"]:
        errores.append("Edad fuera de rango (18-120)")
    if not row["val_lat_rango"]:
        errores.append("Latitud fuera de rango (-90 a 90)")
    if not row["val_long_rango"]:
        errores.append("Longitud fuera de rango (-180 a 180)")

    return " | ".join(errores) if errores else "Registro válido"

# Generar observaciones para todos los registros
df_validacion["observaciones"] = df_validacion.apply(obtener_observaciones, axis=1)

# Mostrar registros con errores
df_validacion[["observaciones", "registro_valido"]].head(10)

,observaciones,registro_valido
0,Registro válido,True
1,Registro válido,True
2,Registro válido,True
3,Registro válido,True
4,Registro válido,True
5,Registro válido,True
6,Registro válido,True
7,Registro válido,True
8,Registro válido,True
9,Registro válido,True


## Separación de registros válidos e inválidos

In [10]:
# Separación de registros válidos e inválidos (caso FRAUDE)
df_validos = df_validacion[df_validacion["registro_valido"]].copy()
df_invalidos = df_validacion[~df_validacion["registro_valido"]].copy()

print("Cantidad de registros totales:", len(df_validacion))
print("Cantidad de registros válidos:", len(df_validos))
print("Cantidad de registros inválidos:", len(df_invalidos))

# Mostrar las observaciones más comunes en inválidos
if len(df_invalidos) > 0:
    print('\nTop motivos de invalidez:')
    print(df_invalidos['observaciones'].value_counts().head(10))
else:
    print('\nNo se encontraron registros inválidos.')

Cantidad de registros totales: 555719
Cantidad de registros válidos: 549427
Cantidad de registros inválidos: 6292

Top motivos de invalidez:
observaciones
Edad fuera de rango (18-120)    6292
Name: count, dtype: int64


In [ ]:
# Añadir metadatos y limpiar columnas relevantes para el caso de fraude

df_validos = df_validos.copy()
df_invalidos = df_invalidos.copy()

# Estado y origen
df_validos["estado_calidad"] = "valido"
df_invalidos["estado_calidad"] = "error"

df_validos["fuente"] = "python_notebook"
df_invalidos["fuente"] = "python_notebook"

# Copiar observaciones al campo estándar 'observacion'
df_invalidos["observacion"] = df_invalidos["observaciones"]

# Asegurar tipos numéricos relevantes: 'amt' y 'city_pop' cuando existan
if 'amt' in df_validos.columns:
    df_validos['amt'] = pd.to_numeric(df_validos['amt'], errors='coerce')
    df_invalidos['amt'] = pd.to_numeric(df_invalidos['amt'], errors='coerce')

if 'city_pop' in df_validos.columns:
    df_validos['city_pop'] = pd.to_numeric(df_validos['city_pop'], errors='coerce')
    df_invalidos['city_pop'] = pd.to_numeric(df_invalidos['city_pop'], errors='coerce')

print('Ejemplos válidos:')
print(df_validos[['amt','registro_valido']].head())

print('\nEjemplos inválidos (observaciones):')
print(df_invalidos[['observaciones','registro_valido']].head())

Ejemplos válidos:
     amt  registro_valido
0   2.86             True
1  29.84             True
2  41.28             True
3  60.05             True
4   3.19             True

Ejemplos inválidos (observaciones):
                    observaciones  registro_valido
112  Edad fuera de rango (18-120)            False
142  Edad fuera de rango (18-120)            False
208  Edad fuera de rango (18-120)            False
272  Edad fuera de rango (18-120)            False
432  Edad fuera de rango (18-120)            False


## Registro del proceso en logs

In [ ]:
logging.info(f"Total registros procesados: {len(df_validacion)}")
logging.info(f"Total registros válidos: {len(df_validos)}")
logging.info(f"Total registros inválidos: {len(df_invalidos)}")

if len(df_invalidos) > 0:
    # Top motivos
    top_reasons = df_invalidos['observaciones'].value_counts().head(10)
    logging.info("Top motivos de invalidez:\n" + top_reasons.to_string())

    # Muestras de registros inválidos con columnas clave si existen
    sample_cols = ['trans_num', 'cc_num', 'amt', 'trans_date_trans_time', 'observaciones']
    sample_cols = [c for c in sample_cols if c in df_invalidos.columns]
    for idx, row in df_invalidos[sample_cols].head(10).iterrows():
        logging.warning(f"Registro inválido index={idx}: " + ", ".join([f"{c}={row[c]}" for c in sample_cols]))

logging.info('Registro de validación completado')
print("Información registrada en ../logs/validation.log")

Información registrada en ../logs/validation.log


## Exportación de resultados

Los archivos generados en esta etapa quedarán en la carpeta `processed`, ya que son resultados del pipeline listos para la siguiente fase.

In [14]:
os.makedirs("../data/processed", exist_ok=True)

valid_path = "../data/processed/fraude_validas.csv"
invalid_path = "../data/processed/fraude_invalidas.csv"

# Guardar sin índices
df_validos.to_csv(valid_path, index=False)
df_invalidos.to_csv(invalid_path, index=False)

print(f"Archivos exportados correctamente:\n - {valid_path}\n - {invalid_path}")

Archivos exportados correctamente:
 - ../data/processed/fraude_validas.csv
 - ../data/processed/fraude_invalidas.csv
